# 🔍 Detecção de Anomalias em Transações Bancárias

**Bootcamp Bradesco - GenAI, Dados & Cyber** | **DIO**

---

## 📋 Sobre o Projeto

Este notebook implementa um sistema de **detecção de fraudes em transações financeiras** usando Machine Learning.

**Objetivo:** Identificar transações suspeitas/fraudulentas em um dataset real, similar aos sistemas antifraude usados por bancos como o Bradesco.

**Dataset:** [Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud) - transações reais de cartão de crédito ocorridas em setembro de 2013 na Europa.

## 1. Instalação e Importações

In [ ]:
# Instalação de dependências (caso necessário)
!pip install -q pandas numpy matplotlib seaborn scikit-learn imbalanced-learn

# Importações
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, roc_curve, precision_recall_curve)

from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE

import warnings
warnings.filterwarnings('ignore')

print('✅ Bibliotecas carregadas com sucesso!')

## 2. Carregamento dos Dados

O dataset contém transações realizadas por cartões de crédito em setembro de 2013 por titulares de cartão europeus.

**Features:**
- `Time`, `V1` a `V28` (componentes PCA - anonimizadas)
- `Amount` (valor da transação)
- `Class` (0 = normal, 1 = fraude)

In [ ]:
# Download do dataset via URL pública
!wget -q -O creditcard.csv "https://github.com/nsethi31/Kaggle-Data-Credit-Card-Fraud-Detection/raw/master/creditcard.csv"

# Carrega o dataset
df = pd.read_csv('creditcard.csv')

print(f'✅ Dataset carregado: {df.shape[0]:,} linhas x {df.shape[1]} colunas')

In [ ]:
# Primeiras linhas
df.head()

In [ ]:
# Informações gerais
df.info()

## 3. Análise Exploratória dos Dados (EDA)

In [ ]:
# Distribuição das classes
class_counts = df['Class'].value_counts()
class_pct = df['Class'].value_counts(normalize=True) * 100

print('=== Distribuição das Classes ===')
print(f'Transações Normais (0): {class_counts[0]:,} ({class_pct[0]:.4f}%)')
print(f'Fraudes (1):          {class_counts[1]:,} ({class_pct[1]:.4f}%)')
print(f'\n📊 Proporção: 1 fraude para cada {class_counts[0]//class_counts[1]:,} transações normais')

# Gráfico
plt.figure(figsize=(8, 5))
colors = ['#4CAF50', '#F44336']
bars = plt.bar(['Normal (0)', 'Fraude (1)'], class_counts, color=colors, edgecolor='black')
plt.ylabel('Quantidade de Transações')
plt.title('Distribuição das Classes', fontsize=14, fontweight='bold')
for bar, val in zip(bars, class_counts):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 500,
             f'{val:,}', ha='center', fontweight='bold')
plt.yscale('log')
plt.show()

print('\n⚠️ Dataset extremamente desbalanceado! Técnicas de balanceamento serão necessárias.')

In [ ]:
# Análise do valor das transações
print('=== Estatísticas do Valor (Amount) ===')
print(df.groupby('Class')['Amount'].describe().round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Boxplot
sns.boxplot(data=df, x='Class', y='Amount', palette=colors, ax=axes[0])
axes[0].set_title('Distribuição de Valores por Classe')
axes[0].set_xticklabels(['Normal', 'Fraude'])

# Histograma
for cls, color, label in [(0, '#4CAF50', 'Normal'), (1, '#F44336', 'Fraude')]:
    subset = df[df['Class'] == cls]['Amount']
    axes[1].hist(subset, bins=50, alpha=0.6, color=color, label=label, density=True)
axes[1].set_title('Distribuição de Valores (densidade)')
axes[1].set_xlabel('Valor da Transação')
axes[1].legend()
axes[1].set_xlim(0, 500)

plt.tight_layout()
plt.show()

In [ ]:
# Análise temporal
plt.figure(figsize=(14, 5))
for cls, color, label in [(0, '#4CAF50', 'Normal'), (1, '#F44336', 'Fraude')]:
    subset = df[df['Class'] == cls]
    plt.hist(subset['Time'], bins=48, alpha=0.5, color=color, label=label, density=True)
plt.xlabel('Tempo (segundos desde primeira transação)')
plt.ylabel('Densidade')
plt.title('Distribuição Temporal das Transações', fontweight='bold')
plt.legend()
plt.show()

In [ ]:
# Correlação das features com a classe alvo
corr_with_class = df.drop(columns=['Time']).corr()['Class'].drop('Class').sort_values()

plt.figure(figsize=(10, 8))
colors_corr = ['#F44336' if v < 0 else '#4CAF50' for v in corr_with_class.values]
corr_with_class.plot(kind='barh', color=colors_corr)
plt.title('Correlação das Features com a Classe (Fraude)', fontweight='bold')
plt.xlabel('Correlação')
plt.axvline(0, color='black', linestyle='-', linewidth=0.5)
plt.tight_layout()
plt.show()

## 4. Pré-processamento

### 4.1 Padronização da coluna Amount

A coluna `Amount` está em escala diferente das features PCA. Vamos padronizá-la com StandardScaler.

In [ ]:
# Padroniza Amount e Time
scaler = StandardScaler()
df['Amount_Scaled'] = scaler.fit_transform(df[['Amount']])
df['Time_Scaled'] = scaler.fit_transform(df[['Time']])

# Prepara features e target
features = ['Amount_Scaled', 'Time_Scaled'] + [f'V{i}' for i in range(1, 29)]
X = df[features]
y = df['Class']

print(f'✅ Features: {X.shape[1]} colunas')
print(f'✅ Target: {y.name} ({y.nunique()} classes)')

In [ ]:
# Divisão treino/teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f'Treino: {X_train.shape[0]:,} amostras')
print(f'Teste:  {X_test.shape[0]:,} amostras')
print(f'\nFraudes no treino: {y_train.sum():,}')
print(f'Fraudes no teste:  {y_test.sum():,}')

### 4.2 Balanceamento das Classes

Como o dataset é extremamente desbalanceado (~0.17% de fraudes), vamos usar **SMOTE** (Synthetic Minority Oversampling Technique) para gerar amostras sintéticas da classe minoritária.

In [ ]:
# SMOTE para balancear as classes
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print('=== Balanceamento com SMOTE ===')
print(f'Antes: {y_train.value_counts().to_dict()}')
print(f'Depois: {pd.Series(y_train_bal).value_counts().to_dict()}')

## 5. Modelagem

Vamos treinar dois modelos:
1. **Regressão Logística** — modelo linear, rápido e interpretável
2. **Random Forest** — modelo ensemble, mais robusto para dados complexos

In [ ]:
# === Modelo 1: Regressão Logística ===
print('🔄 Treinando Regressão Logística...')
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_bal, y_train_bal)
y_pred_lr = lr.predict(X_test)
y_prob_lr = lr.predict_proba(X_test)[:, 1]

print('✅ Regressão Logística treinada!')

In [ ]:
# === Modelo 2: Random Forest ===
print('🔄 Treinando Random Forest (pode levar alguns segundos)...')
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf.fit(X_train_bal, y_train_bal)
y_pred_rf = rf.predict(X_test)
y_prob_rf = rf.predict_proba(X_test)[:, 1]

print('✅ Random Forest treinada!')

## 6. Avaliação dos Resultados

Para detecção de fraudes, as métricas mais importantes são:
- **Recall** (taxa de detecção de fraudes) — quantas fraudes conseguimos identificar?
- **Precisão** — das que marcamos como fraude, quantas são realmente?
- **F1-Score** — equilíbrio entre precisão e recall
- **AUC-ROC** — capacidade do modelo de distinguir as classes

In [ ]:
def plot_confusion_matrix(y_true, y_pred, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Normal', 'Fraude'],
                yticklabels=['Normal', 'Fraude'])
    plt.title(f'Matriz de Confusão — {title}', fontweight='bold')
    plt.ylabel('Real')
    plt.xlabel('Predito')
    plt.show()

print('='*60)
print('📊 REGRESSÃO LOGÍSTICA')
print('='*60)
plot_confusion_matrix(y_test, y_pred_lr, 'Regressão Logística')
print(classification_report(y_test, y_pred_lr, target_names=['Normal', 'Fraude']))
print(f'AUC-ROC: {roc_auc_score(y_test, y_prob_lr):.4f}')

print('\n' + '='*60)
print('🌲 RANDOM FOREST')
print('='*60)
plot_confusion_matrix(y_test, y_pred_rf, 'Random Forest')
print(classification_report(y_test, y_pred_rf, target_names=['Normal', 'Fraude']))
print(f'AUC-ROC: {roc_auc_score(y_test, y_prob_rf):.4f}')

In [ ]:
# Curva ROC comparativa
plt.figure(figsize=(10, 7))

for model_name, y_prob in [('Regressão Logística', y_prob_lr), ('Random Forest', y_prob_rf)]:
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, label=f'{model_name} (AUC = {auc:.4f})', linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Aleatório')
plt.xlabel('Taxa de Falsos Positivos (FPR)', fontsize=12)
plt.ylabel('Taxa de Verdadeiros Positivos (TPR)', fontsize=12)
plt.title('Curva ROC — Comparação dos Modelos', fontweight='bold', fontsize=14)
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Features mais importantes (Random Forest)
importance = pd.DataFrame({
    'feature': features,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False).head(10)

plt.figure(figsize=(10, 6))
sns.barplot(data=importance, x='importance', y='feature', palette='viridis')
plt.title('Top 10 Features Mais Importantes (Random Forest)', fontweight='bold')
plt.xlabel('Importância')
plt.tight_layout()
plt.show()

## 7. Conclusão

### Resultados Obtidos

| Modelo | Acurácia | Precision (Fraude) | Recall (Fraude) | F1 (Fraude) | AUC-ROC |
|--------|----------|--------------------|-----------------|-------------|---------|
| Regressão Logística | | | | | |
| Random Forest | | | | | |

> *Os valores serão preenchidos após a execução do notebook.*

### Insights

- O **desbalanceamento extremo** (~0.17% de fraudes) é o principal desafio — resolvido com SMOTE
- **Random Forest** tende a performar melhor que Regressão Logística por capturar relações não-lineares
- As features **V14, V17, V10 e V12** são as mais correlacionadas com fraudes
- O **Recall** é a métrica mais crítica: perder uma fraude (falso negativo) é muito pior que um falso alarme

### Próximos Passos

1. Testar outros modelos (XGBoost, LightGBM)
2. Ajustar hiperparâmetros com GridSearchCV
3. Implementar threshold tuning para otimizar recall
4. Simular detecção em tempo real com dados novos